# Flight simulator

# Imports

In [ ]:
import json
import os
from dataclasses import asdict, replace

os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image

from uavloc.data.rasters import load_raster
from uavloc.features.factory import build_features, correlator_config
from uavloc.geo import LatLon
from uavloc.matching.correlator import PositionCorrelator
from uavloc.plotting.flight import plot_errors, plot_tracks
from uavloc.plotting.flight_video import FlightVideo, VideoConfig
from uavloc.simulation.camera import CameraConfig, RasterCamera
from uavloc.simulation.correctors import Corrector, CorrelatorCorrector, InsOnly
from uavloc.simulation.flight import FlightConfig, flight_summary, simulate_flight
from uavloc.simulation.ins import InsConfig, InsSimulator
from uavloc.simulation.route import check_inside, random_waypoints, route_length_m, sample_route, waypoints_from_fractions
from uavloc.track import REPO_ROOT, set_seed

# Params

In [ ]:
RUN_NAME = "bushcha-loop_80m_30fps"
OUT_DIR = "tools/flight_simulator/outputs"      # relative to src/; one folder per RUN_NAME
SEED = 42
DEVICE = "cpu"                                  # "cpu" | "mps"

# imagery: what the drone camera sees, and the onboard map it is matched against
PLACE = "bushcha-50p2422-25p8305"             # seg models trained on dubrova + birky below row 704; all other villages unseen
DRONE_SOURCE = {"kind": "multidate", "place": PLACE, "date": "current"}
MAP_SOURCE = {"kind": "multidate", "place": PLACE, "date": "newest"}   # "oldest" | "newest" | a date on disk
# any georeferenced image: {"kind": "file", "path": "data/derived/<...>.jpg", "top_left": [lat, lon], "bottom_right": [lat, lon]}

# route
ROUTE_KIND = "fractions"                        # "fractions" | "latlon" | "random"
ROUTE_FRACTIONS = [(0.30, 0.35), (0.65, 0.40), (0.60, 0.65), (0.35, 0.60)]   # (x, y) share of map width / height, from top-left
ROUTE_LATLON = []                               # [(lat, lon), ...]
RANDOM_WAYPOINTS = 5
ROUTE_EDGE_MARGIN_M = 60.0                      # poses keep this far from both rasters' edges

# flight
ALTITUDE_M = 80.0
SPEED_MPS = 10.0
STEP_S = 0.5                                    # pose spacing; one video frame per pose
CORRECTION_INTERVAL_S = 5.0
FOV_W_DEG = 60.0
CAMERA_ASPECT_W_OVER_H = 16 / 9
DRONE_VIEW_WIDTH_PX = None                      # None: raster resolution (what the corrector sees)

# INS drift
INS_VELOCITY_BIAS_MPS = 0.3                     # std per axis, drawn once per flight
INS_VELOCITY_RANDOM_WALK_MPS = 0.02             # std per axis of the bias change per second
INS_HEADING_BIAS_DEG = 2.0                      # std of a constant heading error
INS_SPEED_SCALE_STD = 0.01
INS_REPORTED_YAW_NOISE_DEG = 0.0                # extra noise on the yaw handed to the corrector
INS_REPORTED_ALTITUDE_NOISE_M = 0.0
INS_INITIAL_ERROR_M = 0.0

# corrector
CORRECTOR = "correlator"                        # "correlator" | "ins-only"
METHOD = "segmentation"                         # "segmentation" | "vgg11"
SEG_ARCHITECTURE = "lraspp"                     # "lraspp" | "unet"
SEG_CHECKPOINT_DRONE = "pocs/mobilenet-lraspp-speed/model-lraspp-bn-satellite.pt"
SEG_CHECKPOINT_SAT = "pocs/mobilenet-lraspp-speed/model-lraspp-bn-satellite.pt"
# U-Net round 3: "pocs/village-seg-baseline/model-round3-bn-satellite.pt" with SEG_ARCHITECTURE = "unet"
SEG_CHANNELS = "all"                            # "all" | "no-background"
DRONE_SEG_GSD_M = 0.385                         # GSD the drone-side model expects
MATCH_GSD_M = 0.385
VGG11_CHECKPOINT = "../archive/uav_localization_2025/feature_models/models_finetrain/vgg11_drop_margin08_adamw_model_20250521_034900.pth"
DRONE_INPUT_PX = 448
MATCHING = "weighted-ncc"                       # "weighted-ncc" | "cosine" (cosine for vgg11)
CLASS_WEIGHTS = [0.0, 1.0, 1.0, 0.5, 0.0, 0.5]  # background, building, track, woody, field, water
WINDOW = "rect"                                 # "rect" | "round"
SEARCH_SCALE = 1.7                              # rect: search patch = scale x footprint
SEARCH_RADIUS_M = None                          # round: how far the fix may move
PRIOR_SIGMA_M = None                            # None: flat prior
TEMPERATURE = 1.0
GATE = "none"                                   # "none" | "full" | "accept-reject"
GATE_S_MAX_M = None                             # e.g. 3.0 with GATE = "full"
GATE_C_MIN = None                               # e.g. 0.01 with GATE = "full"
SAT_FEATURE_CROP_PCT = 0.03
DRONE_FEATURE_CROP_PCT = 0.0

# video
SAVE_VIDEO = True
VIDEO_FPS = 30.0
VIDEO_SIZE_PX = (1280, 720)
VIDEO_MAIN_PANEL = "drone"                      # "drone" | "map" | "segmentation" | "heatmap"
VIDEO_CORNER_PANELS = {"top-right": "segmentation", "bottom-right": "map"}   # corners: top/bottom-left/right
VIDEO_PANEL_WIDTH_SHARE = 0.3
VIDEO_MAP_MODE = "follow"                       # "follow" | "route"
VIDEO_MAP_RADIUS_M = 120.0                      # follow mode: half the map panel's width
VIDEO_TRACKS = ["truth", "ins", "estimate"]
VIDEO_TRAIL_S = None                            # None: whole track so far
VIDEO_SEGMENTATION = "last-fix"                 # "last-fix" | "every-frame" (runs the model on every frame)
VIDEO_OVERLAY_ALPHA = 0.0                       # segmentation over the drone view, 0..1
VIDEO_STEP_STRIDE = 1
VIDEO_CODEC = "avc1"                            # "mp4v" if avc1 fails
VIDEO_VIEW_WIDTH_PX = 1280
VIDEO_PREVIEW_STEP = 60

# Data

In [ ]:
set_seed(SEED)

In [ ]:
out_dir = REPO_ROOT / OUT_DIR / RUN_NAME
out_dir.mkdir(parents=True, exist_ok=True)
out_dir

In [ ]:
drone_raster = load_raster(DRONE_SOURCE)
map_raster = load_raster(MAP_SOURCE)
{"drone px": drone_raster.pixels.shape, "map px": map_raster.pixels.shape, "map m/px": round(map_raster.mpp, 3)}

In [ ]:
def route_waypoints() -> list[LatLon]:
    if ROUTE_KIND == "fractions":
        return waypoints_from_fractions(map_raster, ROUTE_FRACTIONS)
    if ROUTE_KIND == "latlon":
        return [LatLon(lat, lon) for lat, lon in ROUTE_LATLON]
    if ROUTE_KIND == "random":
        return random_waypoints(map_raster, RANDOM_WAYPOINTS, ROUTE_EDGE_MARGIN_M, np.random.default_rng(SEED))
    raise ValueError(f"unknown ROUTE_KIND {ROUTE_KIND!r}")

In [ ]:
waypoints = route_waypoints()
poses = sample_route(waypoints, SPEED_MPS, STEP_S, ALTITUDE_M)

In [ ]:
check_inside(poses, drone_raster, ROUTE_EDGE_MARGIN_M, "drone raster")
check_inside(poses, map_raster, ROUTE_EDGE_MARGIN_M, "map raster")
{"poses": len(poses), "duration_s": poses[-1].t_s, "length_m": round(route_length_m(poses), 1)}

In [ ]:
camera_config = CameraConfig(FOV_W_DEG, CAMERA_ASPECT_W_OVER_H, DRONE_VIEW_WIDTH_PX)
camera = RasterCamera(drone_raster, camera_config)

In [ ]:
fig_start, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].imshow(camera.view(poses[0])), axes[0].set_title("drone view at start")
axes[1].imshow(RasterCamera(map_raster, camera_config).view(poses[0])), axes[1].set_title("map at start")
fig_start.tight_layout()

# Model

In [ ]:
CORRECTOR_CONFIG = {
    "method": METHOD, "seg_architecture": SEG_ARCHITECTURE, "seg_checkpoint_drone": SEG_CHECKPOINT_DRONE,
    "seg_checkpoint_sat": SEG_CHECKPOINT_SAT, "seg_channels": SEG_CHANNELS, "drone_seg_gsd_m": DRONE_SEG_GSD_M,
    "match_gsd_m": MATCH_GSD_M, "vgg11_checkpoint": VGG11_CHECKPOINT, "drone_input_px": DRONE_INPUT_PX,
    "matching": MATCHING, "class_weights": CLASS_WEIGHTS, "window": WINDOW, "search_scale": SEARCH_SCALE,
    "search_radius_m": SEARCH_RADIUS_M, "prior_sigma_m": PRIOR_SIGMA_M, "temperature": TEMPERATURE, "gate": GATE,
    "gate_s_max_m": GATE_S_MAX_M, "gate_c_min": GATE_C_MIN, "fov_w_deg": FOV_W_DEG,
    "sat_feature_crop_pct": SAT_FEATURE_CROP_PCT, "drone_feature_crop_pct": DRONE_FEATURE_CROP_PCT,
}

In [ ]:
def build_corrector() -> Corrector:
    if CORRECTOR == "ins-only":
        return InsOnly()
    features = build_features(CORRECTOR_CONFIG, map_raster, DEVICE)
    config = replace(correlator_config(CORRECTOR_CONFIG), aspect_w_over_h=CAMERA_ASPECT_W_OVER_H)
    return CorrelatorCorrector(PositionCorrelator(features, config), name=f"{METHOD} / {MATCHING}")

In [ ]:
corrector = build_corrector()
corrector.name

# Run

In [ ]:
ins_config = InsConfig(INS_VELOCITY_BIAS_MPS, INS_VELOCITY_RANDOM_WALK_MPS, INS_HEADING_BIAS_DEG, INS_SPEED_SCALE_STD,
                       INS_REPORTED_YAW_NOISE_DEG, INS_REPORTED_ALTITUDE_NOISE_M, INS_INITIAL_ERROR_M)

In [ ]:
result = simulate_flight(poses, camera, InsSimulator(ins_config, SEED), corrector, FlightConfig(CORRECTION_INTERVAL_S))

In [ ]:
summary = flight_summary(result)
pd.Series(summary)

In [ ]:
FIX_COLUMNS = ["t_s", "decision", "ins_error_m", "est_error_m", "peak_z", "s1_m", "s2_m", "fix_ms"]
result.track.loc[result.track["fix_attempted"], FIX_COLUMNS].round(2)

# Plots

In [ ]:
fig_tracks, ax = plt.subplots(figsize=(11, 8))
plot_tracks(result, map_raster, ax)
ax.set_title(f"{RUN_NAME}: {result.corrector_name}, correction every {CORRECTION_INTERVAL_S:g} s")
fig_tracks.tight_layout()

In [ ]:
fig_errors, ax = plt.subplots(figsize=(11, 3.5))
plot_errors(result, ax)
fig_errors.tight_layout()

# Save

In [ ]:
record = {
    "summary": summary, "drone_source": DRONE_SOURCE, "map_source": MAP_SOURCE, "waypoints": [list(w) for w in waypoints],
    "altitude_m": ALTITUDE_M, "speed_mps": SPEED_MPS, "step_s": STEP_S, "correction_interval_s": CORRECTION_INTERVAL_S,
    "camera": asdict(camera_config), "ins": asdict(ins_config), "seed": SEED, "device": DEVICE,
    "corrector": CORRECTOR, "corrector_config": CORRECTOR_CONFIG if CORRECTOR != "ins-only" else None,
}
(out_dir / "run.json").write_text(json.dumps(record, indent=2, default=str))
result.track.to_csv(out_dir / "track.csv", index=False)

In [ ]:
fig_tracks.savefig(out_dir / "tracks.png", dpi=120)
fig_errors.savefig(out_dir / "errors.png", dpi=120)

# Video

In [ ]:
def segment_view(view: Image.Image, width_m: float, height_m: float) -> np.ndarray:
    """Class ids of one video frame from the corrector's drone model (segmentation correctors only)."""
    return corrector.correlator.features.drone_probabilities(view, width_m, height_m).argmax(0)

In [ ]:
video_config = VideoConfig(
    fps=VIDEO_FPS, frame_size_px=tuple(VIDEO_SIZE_PX), main_panel=VIDEO_MAIN_PANEL, corner_panels=VIDEO_CORNER_PANELS,
    panel_width_share=VIDEO_PANEL_WIDTH_SHARE, map_mode=VIDEO_MAP_MODE, map_radius_m=VIDEO_MAP_RADIUS_M,
    tracks=tuple(VIDEO_TRACKS), trail_s=VIDEO_TRAIL_S, segmentation_source=VIDEO_SEGMENTATION,
    overlay_alpha=VIDEO_OVERLAY_ALPHA, step_stride=VIDEO_STEP_STRIDE, codec=VIDEO_CODEC,
)
video_camera = RasterCamera(drone_raster, replace(camera_config, image_width_px=VIDEO_VIEW_WIDTH_PX))
segmenter = segment_view if VIDEO_SEGMENTATION == "every-frame" else None
video = FlightVideo(result, video_camera, map_raster, video_config, segmenter)

In [ ]:
fig_frame, ax = plt.subplots(figsize=(14, 8))
ax.imshow(video.frame(min(VIDEO_PREVIEW_STEP, len(poses) - 1))), ax.axis("off")
fig_frame.tight_layout()

In [ ]:
video.save(out_dir / "flight.mp4") if SAVE_VIDEO else "video skipped"